# Notebook 3 — Model Training

This notebook trains two deep-learning architectures on the FER2013 dataset
and saves the resulting models and training histories to disk.

- **Model A — Custom CNN**: a four-block convolutional network trained from
  scratch on 48 × 48 grayscale images. Serves as the interpretable baseline.
- **Model B — MobileNetV2 fine-tuned**: a lightweight pretrained backbone
  adapted to FER2013 in two stages. Expected to generalise better due to
  ImageNet pretraining.

All evaluation (accuracy curves, confusion matrices, per-class metrics) is
deferred to Notebook 4. Here we only run a brief smoke test after each model
is saved to confirm the output pipeline works end-to-end.

## Section 1 — Setup

We fix all random seeds before any other operation to ensure reproducibility
across runs. Seeds are set for Python's `random` module, NumPy, and TensorFlow
independently because each maintains its own PRNG state. We also verify GPU
availability here; training without a GPU is possible but wall-clock time
increases roughly 10×.

In [ ]:
import json
import random
import sys
from pathlib import Path

import numpy as np
import tensorflow as tf

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import config
from src.models.cnn_custom import build_cnn_custom
from src.models.mobilenet_finetune import build_mobilenet_head, unfreeze_top_layers
from src.models.trainer import get_default_callbacks, save_history, train_model

SEED = config.RANDOM_SEED
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

gpus = tf.config.list_physical_devices("GPU")
if gpus:
    print(f"GPU available: {gpus}")
else:
    print("WARNING: No GPU detected. Training will be significantly slower.")

## Section 2 — Load preprocessed data

We reload the `.npz` files and class-weights JSON produced by Notebook 2
rather than re-running the preprocessing pipeline. This ensures that the
exact same normalised tensors and train/val/test split are used for both
models, and reduces notebook startup time from several minutes to a few
seconds. JSON dictionary keys are strings by default; we cast them to `int`
because the Keras `class_weight` argument requires integer keys.

In [ ]:
gray = np.load(config.PROCESSED_DIR / "fer2013_gray.npz")
rgb  = np.load(config.PROCESSED_DIR / "fer2013_rgb64.npz")

with open(config.PROCESSED_DIR / "class_weights.json") as f:
    class_weights_raw = json.load(f)
class_weights = {int(k): v for k, v in class_weights_raw.items()}

X_train_gray, y_train = gray["X_train"], gray["y_train"]
X_val_gray,   y_val   = gray["X_val"],   gray["y_val"]
X_test_gray,  y_test  = gray["X_test"],  gray["y_test"]

X_train_rgb = rgb["X_train"]
X_val_rgb   = rgb["X_val"]
X_test_rgb  = rgb["X_test"]

print(f"Grayscale  — train: {X_train_gray.shape}  val: {X_val_gray.shape}  test: {X_test_gray.shape}")
print(f"RGB 64×64  — train: {X_train_rgb.shape}  val: {X_val_rgb.shape}  test: {X_test_rgb.shape}")
print(f"Labels     — y_train: {y_train.shape}  y_val: {y_val.shape}  y_test: {y_test.shape}")
print(f"Class weights: {class_weights}")

## Section 3 — Custom CNN (Model A)

### Section 3.1 — Build the architecture

The custom CNN uses four convolutional blocks with doubling filter counts
(32 → 64 → 128 → 256), each followed by BatchNorm, ReLU, MaxPooling, and
Dropout(0.25). Two dense layers (512, 256) with Dropout(0.5) precede the
7-class softmax head. This depth balances representational capacity against
overfitting risk on 48 × 48 images — a fifth block would multiply parameters
with minimal feature-resolution gain given the small spatial size.

In [ ]:
model_cnn = build_cnn_custom(input_shape=(48, 48, 1), num_classes=7)
model_cnn.summary()

### Section 3.2 — Configure training callbacks

Three standard callbacks are attached. `EarlyStopping` (patience=7) halts
training when validation loss stagnates and restores the best weights seen
during the run. `ReduceLROnPlateau` (patience=3, factor=0.5) halves the
learning rate after three epochs without improvement, letting the optimiser
converge into narrow loss minima that a large fixed LR would overshoot.
`ModelCheckpoint` writes the best checkpoint independently, so it is
recoverable even if the process is interrupted.

In [ ]:
config.MODELS_DIR.mkdir(parents=True, exist_ok=True)
config.HISTORIES_DIR.mkdir(parents=True, exist_ok=True)

callbacks_cnn = get_default_callbacks("cnn_custom", config.MODELS_DIR)
print("Callbacks configured:")
for cb in callbacks_cnn:
    print(f"  {cb.__class__.__name__}")

### Section 3.3 — Train the custom CNN

Up to 50 epochs with batch size 64; in practice `EarlyStopping` fires
between epochs 20–40 on this dataset. Class weights are passed so the
optimiser penalises misclassifications of rare emotions (e.g. *Disgust*,
~1.5% of samples) at the same effective rate as the dominant class
(*Happy*, ~25%). Expected validation accuracy at convergence: 60–68 %.

In [ ]:
history_cnn = train_model(
    model=model_cnn,
    train_data=(X_train_gray, y_train),
    val_data=(X_val_gray, y_val),
    class_weights=class_weights,
    epochs=50,
    callbacks=callbacks_cnn,
    batch_size=config.BATCH_SIZE,
)
print(f"Training complete — {len(history_cnn.history["loss"])} epochs run.")

### Section 3.4 — Save the custom CNN

Two artefacts are persisted: the final model (used by the inference pipeline
and the API service) and the training history as JSON (used by Notebook 4 to
plot learning curves without retraining). The `.keras` format is preferred
on TF 2.10; if saving fails the cell falls back to `.h5` and logs the reason.
Both model and history paths are printed so they can be verified on disk.

In [ ]:
model_path_cnn = config.MODELS_DIR / "cnn_custom.keras"
try:
    model_cnn.save(str(model_path_cnn))
    print(f"Saved model: {model_path_cnn}")
except Exception as e:
    model_path_cnn = config.MODELS_DIR / "cnn_custom.h5"
    model_cnn.save(str(model_path_cnn))
    print(f"Fell back to .h5 — saved: {model_path_cnn}  (reason: {e})")

history_path_cnn = config.HISTORIES_DIR / "cnn_custom_history.json"
save_history(history_cnn, history_path_cnn)
print(f"Saved history ({len(history_cnn.history["loss"])} epochs): {history_path_cnn}")

## Section 4 — MobileNetV2 Fine-tuned (Model B)

### Section 4.1 — Build the MobileNetV2 head

A small classification head is attached to a frozen MobileNetV2 backbone
loaded with ImageNet weights. At this stage only the
`GlobalAveragePooling2D → Dense(256) → Dense(7)` head is trainable; the
backbone's 154 layers are frozen. The trainable vs non-trainable parameter
counts printed below confirm this — frozen parameters participate in the
forward pass but receive no gradient updates during Stage 1.

In [ ]:
model_mob = build_mobilenet_head(input_shape=(64, 64, 3), num_classes=7)
model_mob.summary()

trainable_params = np.sum([np.prod(v.shape) for v in model_mob.trainable_weights])
frozen_params    = np.sum([np.prod(v.shape) for v in model_mob.non_trainable_weights])
print(f"
Trainable params:  {trainable_params:,}")
print(f"Frozen params:     {frozen_params:,}")

### Section 4.2 — Stage 1: train the classification head

The head is trained for 10 epochs with lr = 1e-3 before any backbone layers
are touched. This warm-up lets the randomly initialised dense layers stabilise
their weights. Without it, large head gradients would corrupt the pretrained
backbone features during the first few backward passes — a form of catastrophic
forgetting caused by the newly initialised layers rather than the fine-tuning
itself.

In [ ]:
callbacks_mob = get_default_callbacks("mobilenet_ft", config.MODELS_DIR)

history_mob_s1 = train_model(
    model=model_mob,
    train_data=(X_train_rgb, y_train),
    val_data=(X_val_rgb, y_val),
    class_weights=class_weights,
    epochs=10,
    callbacks=callbacks_mob,
    batch_size=config.BATCH_SIZE,
)
print(f"
Stage 1 done — final val_accuracy: {history_mob_s1.history["val_accuracy"][-1]:.4f}")

### Section 4.3 — Stage 2: unfreeze top layers and fine-tune

The top 30 backbone layers are unfrozen and the model recompiled with
lr = 1e-5. The very low learning rate is essential: FER2013 is a small,
domain-shifted dataset compared to ImageNet, so large gradients would
overwrite the pretrained spatial-feature detectors that make transfer
learning useful. The second parameter count confirms that trainable params
increase substantially after the unfreeze.

In [ ]:
model_mob = unfreeze_top_layers(model_mob, n_layers=30)

trainable_ft = np.sum([np.prod(v.shape) for v in model_mob.trainable_weights])
frozen_ft    = np.sum([np.prod(v.shape) for v in model_mob.non_trainable_weights])
print(f"Trainable params after unfreeze: {trainable_ft:,}")
print(f"Frozen params after unfreeze:    {frozen_ft:,}")

history_mob_s2 = train_model(
    model=model_mob,
    train_data=(X_train_rgb, y_train),
    val_data=(X_val_rgb, y_val),
    class_weights=class_weights,
    epochs=20,
    callbacks=callbacks_mob,
    batch_size=config.BATCH_SIZE,
)
print(f"
Stage 2 done — final val_accuracy: {history_mob_s2.history["val_accuracy"][-1]:.4f}")

### Section 4.4 — Save the fine-tuned MobileNetV2

The Stage 1 and Stage 2 histories are concatenated before saving: for each
metric key, the two value lists are appended to form a single sequence
covering all training epochs. This gives Notebook 4 one continuous learning
curve per metric without needing to know about the two-stage protocol.
The total epoch count printed below should equal Stage 1 + Stage 2 epochs
that actually ran (early stopping may reduce the Stage 2 count).

In [ ]:
model_path_mob = config.MODELS_DIR / "mobilenet_ft.keras"
try:
    model_mob.save(str(model_path_mob))
    print(f"Saved model: {model_path_mob}")
except Exception as e:
    model_path_mob = config.MODELS_DIR / "mobilenet_ft.h5"
    model_mob.save(str(model_path_mob))
    print(f"Fell back to .h5 — saved: {model_path_mob}  (reason: {e})")

combined_history: dict = {
    key: history_mob_s1.history[key] + history_mob_s2.history[key]
    for key in history_mob_s1.history
}
history_path_mob = config.HISTORIES_DIR / "mobilenet_ft_history.json"
serializable = {k: [float(v) for v in vals] for k, vals in combined_history.items()}
with open(history_path_mob, "w", encoding="utf-8") as f:
    json.dump(serializable, f, indent=2)
total_epochs = len(combined_history["loss"])
print(f"Saved combined history ({total_epochs} epochs total): {history_path_mob}")

## Section 5 — Quick sanity check

Both models predict the emotion label for the first 16 test images. The goal
is not to measure accuracy (that is the subject of Notebook 4) but to confirm
that each model produces a valid 7-class output and that the argmax pipeline
works correctly. A model that predicts the same class for every input would
signal a training failure worth investigating before proceeding.

In [ ]:
LABELS = config.EMOTION_LABELS

def _smoke_test(model, X_subset, y_subset, name):
    preds = model.predict(X_subset, verbose=0)
    pred_cls = np.argmax(preds, axis=1)
    true_cls = np.argmax(y_subset, axis=1)
    print(f"
{name} — predictions on first 16 test samples:")
    print(f"  {'#':>2}  {'Predicted':<12} {'True':<12} Match")
    print("  " + "-" * 38)
    for i, (p, t) in enumerate(zip(pred_cls, true_cls)):
        mark = "v" if p == t else "x"
        print(f"  {i:>2}  {LABELS[p]:<12} {LABELS[t]:<12} {mark}")

_smoke_test(model_cnn, X_test_gray[:16], y_test[:16], "Custom CNN")
_smoke_test(model_mob, X_test_rgb[:16],  y_test[:16], "MobileNetV2")